# Practical Session 02
## Part I - Instructor-led mini-lab
### Example 1 - A small numerical experiment object

We start with a Gaussian wave packet

$$
\psi(x)=\exp\left[-\left(\frac{x-x_0}{\sigma}\right)^2\right]\exp(ik_0x),
\qquad
\rho(x)=|\psi(x)|^2.
$$

Before plotting, think about the following questions:

1. What is the shape of `psi` and `rho`?
2. Which array stores the coordinate information?
3. Which diagnostics should be computed before looking at the figure?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
x = np.linspace(-8.0, 8.0, 2000)

x0 = 0.6
sigma = 1.2
k0 = 8.0

psi = np.exp(-((x - x0) / sigma)**2) * np.exp(1j * k0 * x)
rho = np.abs(psi)**2

print("x.shape:  ", x.shape)
print("psi.shape:", psi.shape)
print("rho.shape:", rho.shape)

assert x.ndim == 1
assert psi.shape == x.shape
assert rho.shape == x.shape
assert np.all(np.diff(x) > 0.0)
assert np.isfinite(rho).all()
assert np.all(rho >= 0.0)

The array contracts above are not only programming checks. They say that the field lives on an ordered one-dimensional grid and that the density is finite and non-negative.

In [ ]:
norm = np.trapezoid(rho, x)

x_mean = np.trapezoid(x * rho, x) / norm
width = np.sqrt(np.trapezoid((x - x_mean)**2 * rho, x) / norm)

imax = np.argmax(rho)
x_peak = x[imax]

diagnostics = {
    "norm": norm,
    "x_mean": x_mean,
    "width": width,
    "x_peak": x_peak,
}

for name, value in diagnostics.items():
    print(f"{name:>10s} = {value:.6g}")

Now use the diagnostics as part of the plot, not as separate information.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.2))

ax.plot(x, rho, label=r"$|\psi(x)|^2$")
ax.axvline(x_mean, linestyle="--", label="centre")

left = x_mean - width
right = x_mean + width
ax.axvspan(left, right, alpha=0.2, label="one width")

ax.plot(x_peak, rho[imax], "o", label="peak")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"density")
ax.set_title("Wave packet density with numerical diagnostics")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()

Questions:

1. Does the visual centre agree with `x_mean`?
2. Does `x_peak` agree with the expected location of the maximum?
3. Which part of the plot would become misleading if the `x` array were not passed to `ax.plot`?

## Example 2 - Data, model, residual

A model may look reasonable on top of noisy data while the residual already shows a systematic problem. We generate synthetic data from a known signal and compare them with a slightly imperfect model.

In [ ]:
rng = np.random.default_rng(seed=123)

t = np.linspace(0.0, 10.0, 300)

true_signal = np.exp(-0.25 * t) * np.sin(3.0 * t)
sigma_noise = 0.08

measured = true_signal + sigma_noise * rng.normal(size=t.size)

# A slightly wrong model: the frequency is not exactly correct.
model = np.exp(-0.25 * t) * np.sin(3.12 * t)

residual = measured - model
rms_residual = np.sqrt(np.mean(residual**2))

print(f"RMS residual: {rms_residual:.4f}")
print(f"Expected noise scale: {sigma_noise:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(6.0, 4.2))

axes[0].plot(t, measured, ".", markersize=3, label="synthetic measurement")
axes[0].plot(t, model, linewidth=2, label="model")
axes[0].fill_between(
    t,
    model - sigma_noise,
    model + sigma_noise,
    alpha=0.25,
    label=r"$\pm 1\sigma$",
)
axes[0].set_ylabel(r"signal $s(t)$")
axes[0].legend()

axes[1].axhline(0.0, linewidth=1)
axes[1].plot(t, residual, ".", markersize=3)
axes[1].set_xlabel(r"time $t$ [arb. units]")
axes[1].set_ylabel("residual")
axes[1].set_title(f"RMS residual = {rms_residual:.3f}")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()

Questions:

1. Is the residual compatible with unstructured random noise?
2. Which plot makes the wrong frequency easier to notice?
3. Why is the uncertainty band useful even if it is only an approximate model of the noise?

## Example 3 - A two-dimensional field and coordinate conventions

A sampled two-dimensional scalar field has the structure

$$
\rho_{ij}=\rho(x_i,y_j).
$$

We intentionally use an asymmetric Gaussian, because it makes axis mistakes easier to see.

In [ ]:
x2 = np.linspace(-5.0, 5.0, 300)
y2 = np.linspace(-3.0, 3.0, 180)

X2, Y2 = np.meshgrid(x2, y2, indexing="ij")

sigma_x = 1.1
sigma_y = 0.7
x0, y0 = 1.0, -0.4

rho2 = np.exp(
    -(
        (X2 - x0)**2 / (2 * sigma_x**2)
        + (Y2 - y0)**2 / (2 * sigma_y**2)
    )
)

dx2 = x2[1] - x2[0]
dy2 = y2[1] - y2[0]

mass2 = np.sum(rho2) * dx2 * dy2
x_cm = np.sum(X2 * rho2) * dx2 * dy2 / mass2
y_cm = np.sum(Y2 * rho2) * dx2 * dy2 / mass2

imax, jmax = np.unravel_index(np.argmax(rho2), rho2.shape)
x_peak, y_peak = x2[imax], y2[jmax]

print("rho2.shape:", rho2.shape)
print("centre of mass:", x_cm, y_cm)
print("peak location: ", x_peak, y_peak)

First display the raw array. This is useful for debugging storage, but the axes are array indices rather than physical coordinates.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 3.4))

im = ax.imshow(rho2)
fig.colorbar(im, ax=ax, label="array value")

ax.set_xlabel("column index")
ax.set_ylabel("row index")
ax.set_title("Raw array display")

fig.tight_layout()

Now plot the same data on the physical grid. The markers should agree with the diagnostic numbers computed above.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))

pcm = ax.pcolormesh(X2, Y2, rho2, shading="auto")

ax.plot(x_peak, y_peak, "o", label="maximum")
ax.plot(x_cm, y_cm, "x", markersize=8, label="centre of mass")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_aspect("equal")
ax.legend()

fig.colorbar(pcm, ax=ax, label=r"$\rho(x,y)$")
fig.tight_layout()

Questions:

1. Why is the raw `imshow` plot insufficient as a scientific field plot?
2. Where would the peak appear if the axes were accidentally exchanged?
3. Why does the asymmetric choice of `sigma_x`, `sigma_y`, `x0`, and `y0` help with debugging?

## Example 4 - A residual map for a differential equation

For

$$
u(x,y)=\sin(kx)\sin(\ell y),
$$

the continuous equation

$$
u_{xx}+u_{yy}+(k^2+\ell^2)u=0
$$

is satisfied exactly. A finite-difference residual checks how well the sampled field satisfies the equation on the chosen grid.

In [ ]:
x_res = np.linspace(0.0, np.pi, 180)
y_res = np.linspace(0.0, np.pi, 140)

Xr, Yr = np.meshgrid(x_res, y_res, indexing="ij")

k = 2.0
ell = 3.0

u = np.sin(k * Xr) * np.sin(ell * Yr)

dx = x_res[1] - x_res[0]
dy = y_res[1] - y_res[0]

uxx = (u[2:, 1:-1] - 2 * u[1:-1, 1:-1] + u[:-2, 1:-1]) / dx**2
uyy = (u[1:-1, 2:] - 2 * u[1:-1, 1:-1] + u[1:-1, :-2]) / dy**2

R = uxx + uyy + (k**2 + ell**2) * u[1:-1, 1:-1]

Xi = Xr[1:-1, 1:-1]
Yi = Yr[1:-1, 1:-1]

res_inf = np.max(np.abs(R))
res_rms = np.sqrt(np.mean(R**2))

print("u.shape:", u.shape)
print("R.shape:", R.shape)
print(f"||R||_inf = {res_inf:.3e}")
print(f"RMS(R)    = {res_rms:.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))

lim = np.max(np.abs(R))
pcm = ax.pcolormesh(
    Xi,
    Yi,
    R,
    shading="auto",
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_aspect("equal")
ax.set_title(fr"Residual map, $\|R\|_\infty={lim:.2e}$")

fig.colorbar(pcm, ax=ax, label="residual")
fig.tight_layout()

Questions:

1. Why is the residual array smaller than the original field?
2. Why is a diverging colour scale appropriate here?
3. What additional information does the residual map provide beyond the scalar norm?

Residual maps are not only useful for PDE solvers. Similar visual diagnostics appear in optimization, inverse problems, finite-element methods, and machine learning.

## Part II - Independent work

### Task 1 - Build a diagnostic packet for a wave packet

Use the following setup:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-10.0, 10.0, 2500)

x0 = -1.3
sigma = 1.4
k0 = 6.5

psi = np.exp(-((x - x0) / sigma)**2) * np.exp(1j * k0 * x)
rho = np.abs(psi)**2

Your tasks are:

1. Create a dictionary called `packet` containing at least `x`, `psi`, `rho`, `dx`, and a short text description.
2. Add numerical contracts checking that:
   - `x` is one-dimensional;
   - `psi` and `rho` have the same shape as `x`;
   - `x` is strictly increasing;
   - `rho` is finite and non-negative.
3. Compute the diagnostics:
   - norm;
   - centre;
   - width;
   - peak position;
   - fraction of the norm inside the window
     $$
     |x-\langle x\rangle|<2\,\mathrm{width}.
     $$
4. Store these diagnostics in a second dictionary called `diagnostics`.
5. Make a diagnostic plot showing `rho`, the centre, one-width interval, and the peak.
6. In two or three sentences, explain which diagnostics would immediately reveal a wrong grid, wrong normalization, or wrong axis scaling.

**Optional extension**: Put the whole construction into a function `wave_packet_experiment(x0, sigma, k0, nx)` returning `packet` and `diagnostics`.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Point 1
# Build a diagnostic packet
# ------------------------------------------------------------

# The coordinate array has shape:
# x[x_index] -> (nx,)
#
# The sampled complex wave function has shape:
# psi[x_index] -> (nx,)
#
# The density has shape:
# rho[x_index] -> (nx,)

dx = x[1] - x[0]

packet = {
    "x": x,
    "psi": psi,
    "rho": rho,
    "dx": dx,
    "description": "Gaussian wave packet on a one-dimensional grid",
}


# ------------------------------------------------------------
# Point 2
# Numerical contracts
# ------------------------------------------------------------

assert x.ndim == 1, f"x must be one-dimensional, but got ndim = {x.ndim}"

assert psi.shape == x.shape, (
    f"psi must have the same shape as x, "
    f"but got psi.shape = {psi.shape} and x.shape = {x.shape}"
)

assert rho.shape == x.shape, (
    f"rho must have the same shape as x, "
    f"but got rho.shape = {rho.shape} and x.shape = {x.shape}"
)

assert np.all(np.diff(x) > 0.0), "x must be strictly increasing"

assert np.isfinite(rho).all(), "rho must contain only finite values"

assert np.all(rho >= 0.0), "rho must be non-negative"

print("x shape:   ", x.shape)
print("psi shape: ", psi.shape)
print("rho shape: ", rho.shape)
print("dx:        ", dx)


# ------------------------------------------------------------
# Point 3
# Compute diagnostics
# ------------------------------------------------------------

# norm = integral rho(x) dx
norm = np.trapezoid(rho, x)

# centre = <x>
centre = np.trapezoid(x * rho, x) / norm

# width = sqrt(<(x - <x>)^2>)
width = np.sqrt(
    np.trapezoid((x - centre)**2 * rho, x) / norm
)

# peak position
peak_index = np.argmax(rho)
peak_position = x[peak_index]
peak_value = rho[peak_index]

# Fraction of norm inside |x - <x>| < 2 * width
window = np.abs(x - centre) < 2.0 * width

norm_inside_window = np.trapezoid(rho[window], x[window])
fraction_inside_window = norm_inside_window / norm


# ------------------------------------------------------------
# Point 4
# Store diagnostics in a dictionary
# ------------------------------------------------------------

diagnostics = {
    "norm": norm,
    "centre": centre,
    "width": width,
    "peak_index": peak_index,
    "peak_position": peak_position,
    "peak_value": peak_value,
    "fraction_inside_2width": fraction_inside_window,
}

print()
print("Diagnostics:")
for name, value in diagnostics.items():
    print(f"{name:>24s}: {value}")


# ------------------------------------------------------------
# Point 5
# Diagnostic plot
# ------------------------------------------------------------

left = centre - width
right = centre + width

fig, ax = plt.subplots(figsize=(7.0, 3.8))

ax.plot(x, rho, label=r"$\rho(x)=|\psi(x)|^2$")

ax.axvline(
    centre,
    linestyle="--",
    label=fr"centre = {centre:.3f}",
)

ax.axvspan(
    left,
    right,
    alpha=0.2,
    label=fr"one width: [{left:.3f}, {right:.3f}]",
)

ax.plot(
    peak_position,
    peak_value,
    "o",
    label=fr"peak at x = {peak_position:.3f}",
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$\rho(x)$")
ax.set_title("Diagnostic plot of a Gaussian wave packet")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


# ------------------------------------------------------------
# Optional extension
# Package the whole experiment into a function
# ------------------------------------------------------------

def wave_packet_experiment(x0, sigma, k0, nx):
    # Check scalar-like input values.
    if sigma <= 0.0:
        raise ValueError(
            f"sigma must be positive, but got sigma = {sigma}"
        )

    if nx < 2:
        raise ValueError(
            f"nx must be at least 2, but got nx = {nx}"
        )

    # Construct the grid.
    x = np.linspace(-10.0, 10.0, nx)
    dx = x[1] - x[0]

    # Construct the wave packet.
    psi = np.exp(-((x - x0) / sigma)**2) * np.exp(1j * k0 * x)
    rho = np.abs(psi)**2

    # Numerical contracts.
    if x.ndim != 1:
        raise ValueError(
            f"x must be one-dimensional, but got x.ndim = {x.ndim}"
        )

    if psi.shape != x.shape:
        raise ValueError(
            f"psi must have shape {x.shape}, but got {psi.shape}"
        )

    if rho.shape != x.shape:
        raise ValueError(
            f"rho must have shape {x.shape}, but got {rho.shape}"
        )

    if not np.all(np.diff(x) > 0.0):
        raise ValueError("x must be strictly increasing")

    if not np.isfinite(rho).all():
        raise ValueError("rho must contain only finite values")

    if not np.all(rho >= 0.0):
        raise ValueError("rho must be non-negative")

    # Store the numerical packet.
    packet = {
        "x": x,
        "psi": psi,
        "rho": rho,
        "dx": dx,
        "description": "Gaussian wave packet on a one-dimensional grid",
    }

    # Compute diagnostics.
    norm = np.trapezoid(rho, x)
    centre = np.trapezoid(x * rho, x) / norm

    width = np.sqrt(
        np.trapezoid((x - centre)**2 * rho, x) / norm
    )

    peak_index = np.argmax(rho)
    peak_position = x[peak_index]
    peak_value = rho[peak_index]

    window = np.abs(x - centre) < 2.0 * width
    norm_inside_window = np.trapezoid(rho[window], x[window])
    fraction_inside_window = norm_inside_window / norm

    diagnostics = {
        "norm": norm,
        "centre": centre,
        "width": width,
        "peak_index": peak_index,
        "peak_position": peak_position,
        "peak_value": peak_value,
        "fraction_inside_2width": fraction_inside_window,
    }

    return packet, diagnostics


# Example use of the optional function.
packet_from_function, diagnostics_from_function = wave_packet_experiment(
    x0=x0,
    sigma=sigma,
    k0=k0,
    nx=2500,
)

assert packet_from_function["x"].shape == x.shape
assert packet_from_function["psi"].shape == psi.shape
assert packet_from_function["rho"].shape == rho.shape

assert np.allclose(packet_from_function["x"], packet["x"])
assert np.allclose(packet_from_function["psi"], packet["psi"])
assert np.allclose(packet_from_function["rho"], packet["rho"])

print()
print("Optional function check passed.")

Short explanation: A wrong grid would usually show up through failed contracts, for example if x is not one-dimensional or not strictly increasing. A wrong normalization would be visible in the norm and in the fraction inside the two-width window, because these numbers summarize the total amount of density. A wrong axis scaling would often be revealed by a mismatch between the diagnostic plot and the numerical values of the centre, width, and peak position.

### Task 2 - Debug a misleading two-dimensional field plot

A colleague generated an asymmetric field, but the first plot is difficult to interpret scientifically.

In [ ]:
x = np.linspace(-4.0, 6.0, 260)
y = np.linspace(-3.0, 4.0, 190)

X, Y = np.meshgrid(x, y, indexing="ij")

field = np.exp(
    -(
        (X - 2.0)**2 / (2 * 0.8**2)
        + (Y + 1.0)**2 / (2 * 1.4**2)
    )
)

# One artificial outlier that should not silently control the whole colour scale.
field_with_outlier = field.copy()
field_with_outlier[25, 150] = 12.0 * field.max()

fig, ax = plt.subplots(figsize=(5.0, 3.4))
im = ax.imshow(field_with_outlier)
fig.colorbar(im, ax=ax)
ax.set_title("First attempt")
plt.show()

Your tasks are:

1. Compute the integral, centre of mass, and maximum location of `field` without the outlier.
2. Make a corrected field plot using physical coordinates. You may use either:
   - `pcolormesh(X, Y, field, shading="auto")`, or
   - `imshow(field.T, extent=[...], origin="lower", aspect="auto")`.
3. Overlay the centre of mass and maximum location on the corrected plot.
4. Set an aspect ratio that preserves the geometry of the field.
5. Make a second plot of `field_with_outlier`, but choose colour limits consciously using the 1st and 99th percentiles.
6. Label both coordinate axes and the colourbar.
7. Conclude with two or three sentences explaining how default plotting choices can hide or invent apparent structure.

**Optional extension**: Compare `imshow` and `pcolormesh` for this example. Which version makes the coordinate convention easier to see?

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Basic numerical contracts
# ------------------------------------------------------------

assert x.ndim == 1, f"x must be one-dimensional, but got x.ndim = {x.ndim}"
assert y.ndim == 1, f"y must be one-dimensional, but got y.ndim = {y.ndim}"

assert np.all(np.diff(x) > 0.0), "x must be strictly increasing"
assert np.all(np.diff(y) > 0.0), "y must be strictly increasing"

assert X.shape == (x.size, y.size)
assert Y.shape == (x.size, y.size)

assert field.shape == (x.size, y.size)
assert field_with_outlier.shape == field.shape

assert np.isfinite(field).all()
assert np.isfinite(field_with_outlier).all()

assert np.all(field >= 0.0)
assert np.all(field_with_outlier >= 0.0)

print("x shape:                  ", x.shape)
print("y shape:                  ", y.shape)
print("field shape:              ", field.shape)
print("field_with_outlier shape: ", field_with_outlier.shape)


# ------------------------------------------------------------
# Point 1
# Compute integral, centre of mass, and maximum location
# for the clean field, without the outlier
# ------------------------------------------------------------

dx = x[1] - x[0]
dy = y[1] - y[0]

# Integral over the two-dimensional domain.
integral = np.sum(field) * dx * dy

# Centre of mass:
# x_cm = integral x * field(x, y) dx dy / integral field(x, y) dx dy
# y_cm = integral y * field(x, y) dx dy / integral field(x, y) dx dy

x_cm = np.sum(X * field) * dx * dy / integral
y_cm = np.sum(Y * field) * dx * dy / integral

# Maximum location.
imax, jmax = np.unravel_index(
    np.argmax(field),
    field.shape,
)

x_peak = x[imax]
y_peak = y[jmax]
field_peak = field[imax, jmax]

print()
print("Clean-field diagnostics:")
print(f"{'integral':>16s}: {integral:.8g}")
print(f"{'x centre':>16s}: {x_cm:.8g}")
print(f"{'y centre':>16s}: {y_cm:.8g}")
print(f"{'x peak':>16s}: {x_peak:.8g}")
print(f"{'y peak':>16s}: {y_peak:.8g}")
print(f"{'peak value':>16s}: {field_peak:.8g}")


# ------------------------------------------------------------
# Point 2, 3, 4, and 6
# Corrected field plot using physical coordinates
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.0, 4.2))

pcm = ax.pcolormesh(
    X,
    Y,
    field,
    shading="auto",
)

# Overlay centre of mass and maximum location.
ax.plot(
    x_cm,
    y_cm,
    "x",
    markersize=9,
    markeredgewidth=2,
    label="centre of mass",
)

ax.plot(
    x_peak,
    y_peak,
    "o",
    markersize=6,
    label="maximum",
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Corrected plot in physical coordinates")

# This preserves the physical geometry of the field.
ax.set_aspect("equal")

cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label("field value")

ax.legend()
fig.tight_layout()
plt.show()


# ------------------------------------------------------------
# Point 5 and 6
# Plot field_with_outlier with conscious colour limits
# ------------------------------------------------------------

vmin, vmax = np.percentile(field_with_outlier, [1.0, 99.0])

print()
print("Colour limits for field_with_outlier:")
print(f"{'1st percentile':>16s}: {vmin:.8g}")
print(f"{'99th percentile':>16s}: {vmax:.8g}")
print(f"{'actual maximum':>16s}: {field_with_outlier.max():.8g}")

fig, ax = plt.subplots(figsize=(6.0, 4.2))

pcm = ax.pcolormesh(
    X,
    Y,
    field_with_outlier,
    shading="auto",
    vmin=vmin,
    vmax=vmax,
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Outlier present, colour scale set by 1st--99th percentiles")
ax.set_aspect("equal")

cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label("field value")

fig.tight_layout()
plt.show()


# ------------------------------------------------------------
# Optional extension
# Compare imshow and pcolormesh for this example
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10.0, 4.0),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

# Use common colour limits for a fair comparison.
vmin = field.min()
vmax = field.max()

# imshow version:
# field has shape (nx, ny), where axis 0 corresponds to x
# and axis 1 corresponds to y.
# imshow displays the first axis vertically and the second axis horizontally,
# so we transpose the field and provide the physical extent.

im = axes[0].imshow(
    field.T,
    extent=[x.min(), x.max(), y.min(), y.max()],
    origin="lower",
    aspect="equal",
    vmin=vmin,
    vmax=vmax,
)

axes[0].plot(x_cm, y_cm, "x", markersize=9, markeredgewidth=2)
axes[0].plot(x_peak, y_peak, "o", markersize=6)
axes[0].set_title("imshow with transpose and extent")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$y$")

# pcolormesh version:
# X and Y appear directly in the plotting command.

pcm = axes[1].pcolormesh(
    X,
    Y,
    field,
    shading="auto",
    vmin=vmin,
    vmax=vmax,
)

axes[1].plot(x_cm, y_cm, "x", markersize=9, markeredgewidth=2)
axes[1].plot(x_peak, y_peak, "o", markersize=6)
axes[1].set_title("pcolormesh with X, Y")
axes[1].set_xlabel(r"$x$")
axes[1].set_aspect("equal")

cbar = fig.colorbar(pcm, ax=axes, label="field value")

plt.show()

Explanation: The default imshow plot shows array row and column indices, not the physical x and y coordinates, so the apparent position and geometry can be misleading.
Automatic colour limits can also let a single outlier control the whole scale, making the main field look artificially flat or hiding relevant structure.
Using physical coordinates, an explicit aspect ratio, labelled axes, and conscious colour limits makes the plot a numerical diagnostic rather than only a picture.


Optional extension comment: For this example, pcolormesh makes the coordinate convention easier to see, because X and Y are passed explicitly to the plotting function.
imshow is also correct, but only after remembering to transpose the field and to set extent and origin consistently.

### Task 3 - Residual maps and convergence as a visual numerical experiment

For

$$
u(x,y)=\sin(kx)\sin(\ell y),
$$

the continuous equation

$$
u_{xx}+u_{yy}+(k^2+\ell^2)u=0
$$

is satisfied exactly. The discrete residual should decrease when the grid is refined.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k = 2.0
ell = 3.0

def make_field(nx, ny):
    x = np.linspace(0.0, np.pi, nx)
    y = np.linspace(0.0, np.pi, ny)
    X, Y = np.meshgrid(x, y, indexing="ij")
    u = np.sin(k * X) * np.sin(ell * Y)
    return x, y, X, Y, u

Your tasks are:

1. Write a function `residual_helmholtz(x, y, u, k, ell)` that returns:
   - the interior coordinate arrays;
   - the residual array;
   - the maximum norm of the residual;
   - the RMS norm of the residual.
2. Use centred finite differences on the interior grid points.
3. Check explicitly that the residual shape is `(nx-2, ny-2)`.
4. For one medium-sized grid, for example `(nx, ny) = (180, 140)`, make:
   - a plot of the field `u`;
   - a signed residual map with a centred diverging colour scale.
5. Repeat the calculation for several grid sizes, for example:
   ```python
   sizes = [(40, 32), (80, 64), (160, 128), (320, 256)]
   ```
6. Store the grid spacing and residual norms in arrays.
7. Make a log-log convergence plot of the maximum residual norm versus grid spacing.
8. Add a reference line proportional to $\Delta x^2$.
9. Estimate the observed order from neighbouring error ratios.
10. Write a short interpretation: does the visual convergence agree with the expected second-order finite-difference stencil?

**Optional extension**: Repeat the test for a higher wavenumber, for example `k = 5`, `ell = 7`. What changes: the order, the error constant, or both?

In [ ]:
#####################
# Your solution here
#####################
# ------------------------------------------------------------
# Point 1 and 2
# Residual for the two-dimensional Helmholtz equation
# ------------------------------------------------------------

def residual_helmholtz(x, y, u, k, ell):
    # Numerical contracts for the grid.
    if x.ndim != 1:
        raise ValueError(
            f"x must be one-dimensional, but got x.ndim = {x.ndim}"
        )

    if y.ndim != 1:
        raise ValueError(
            f"y must be one-dimensional, but got y.ndim = {y.ndim}"
        )

    if x.size < 3:
        raise ValueError(
            f"x must contain at least 3 points, but got x.size = {x.size}"
        )

    if y.size < 3:
        raise ValueError(
            f"y must contain at least 3 points, but got y.size = {y.size}"
        )

    if not np.all(np.diff(x) > 0.0):
        raise ValueError("x must be strictly increasing")

    if not np.all(np.diff(y) > 0.0):
        raise ValueError("y must be strictly increasing")

    if u.shape != (x.size, y.size):
        raise ValueError(
            f"u must have shape ({x.size}, {y.size}), "
            f"but got {u.shape}"
        )

    if not np.isfinite(u).all():
        raise ValueError("u must contain only finite values")

    # Grid spacings.
    dx = x[1] - x[0]
    dy = y[1] - y[0]

    # Interior coordinate arrays.
    Xi, Yi = np.meshgrid(
        x[1:-1],
        y[1:-1],
        indexing="ij",
    )

    # Centred finite differences on the interior grid.
    #
    # u has shape:
    # u[x_index, y_index] -> (nx, ny)
    #
    # u[1:-1, 1:-1] is the interior part:
    # shape -> (nx - 2, ny - 2)

    uxx = (
        u[2:, 1:-1]
        - 2.0 * u[1:-1, 1:-1]
        + u[:-2, 1:-1]
    ) / dx**2

    uyy = (
        u[1:-1, 2:]
        - 2.0 * u[1:-1, 1:-1]
        + u[1:-1, :-2]
    ) / dy**2

    residual = (
        uxx
        + uyy
        + (k**2 + ell**2) * u[1:-1, 1:-1]
    )

    # ------------------------------------------------------------
    # Point 3
    # Check residual shape explicitly
    # ------------------------------------------------------------

    expected_shape = (x.size - 2, y.size - 2)

    assert residual.shape == expected_shape, (
        f"residual should have shape {expected_shape}, "
        f"but got {residual.shape}"
    )

    assert Xi.shape == expected_shape
    assert Yi.shape == expected_shape

    # Residual norms.
    max_norm = np.max(np.abs(residual))
    rms_norm = np.sqrt(np.mean(residual**2))

    return Xi, Yi, residual, max_norm, rms_norm

# ------------------------------------------------------------
# Point 4
# One medium-sized visual numerical experiment
# ------------------------------------------------------------

nx = 180
ny = 140

x, y, X, Y, u = make_field(nx, ny)

Xi, Yi, residual, max_norm, rms_norm = residual_helmholtz(
    x,
    y,
    u,
    k,
    ell,
)

print("Medium-grid diagnostics:")
print("u shape:        ", u.shape)
print("residual shape: ", residual.shape)
print("expected shape: ", (nx - 2, ny - 2))
print(f"max residual:   {max_norm:.8e}")
print(f"RMS residual:   {rms_norm:.8e}")

assert residual.shape == (nx - 2, ny - 2)


# ------------------------------------------------------------
# Plot of the field u
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.8, 4.2))

limit_u = np.max(np.abs(u))

pcm = ax.pcolormesh(
    X,
    Y,
    u,
    shading="auto",
    cmap="RdBu_r",
    vmin=-limit_u,
    vmax=limit_u,
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title(r"Field $u(x,y)=\sin(kx)\sin(\ell y)$")
ax.set_aspect("equal")

cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(r"$u(x,y)$")

fig.tight_layout()
plt.show()


# ------------------------------------------------------------
# Signed residual map with centred diverging colour scale
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.8, 4.2))

res_limit = np.max(np.abs(residual))

pcm = ax.pcolormesh(
    Xi,
    Yi,
    residual,
    shading="auto",
    cmap="RdBu_r",
    vmin=-res_limit,
    vmax=res_limit,
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title(
    fr"Residual map, $\|R\|_\infty = {max_norm:.2e}$"
)
ax.set_aspect("equal")

cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(r"$R(x,y)$")

fig.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Point 5 and 6
# Repeat the calculation for several grid sizes
# ------------------------------------------------------------

sizes = [
    (40, 32),
    (80, 64),
    (160, 128),
    (320, 256),
]

dx_values = []
dy_values = []
h_values = []
max_norms = []
rms_norms = []

for nx, ny in sizes:
    x, y, X, Y, u = make_field(nx, ny)

    Xi, Yi, residual, max_norm, rms_norm = residual_helmholtz(
        x,
        y,
        u,
        k,
        ell,
    )

    dx = x[1] - x[0]
    dy = y[1] - y[0]

    # A single representative grid spacing.
    # Since both directions are refined together, h is enough
    # for the convergence plot.
    h = max(dx, dy)

    dx_values.append(dx)
    dy_values.append(dy)
    h_values.append(h)
    max_norms.append(max_norm)
    rms_norms.append(rms_norm)

    print(
        f"nx = {nx:4d}, ny = {ny:4d}, "
        f"dx = {dx:.4e}, dy = {dy:.4e}, "
        f"max = {max_norm:.4e}, RMS = {rms_norm:.4e}"
    )

dx_values = np.array(dx_values)
dy_values = np.array(dy_values)
h_values = np.array(h_values)
max_norms = np.array(max_norms)
rms_norms = np.array(rms_norms)

In [ ]:
# ------------------------------------------------------------
# Point 7 and 8
# Log-log convergence plot with O(h^2) reference line
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.8, 4.0))

ax.loglog(
    h_values,
    max_norms,
    "o-",
    label=r"measured $\|R\|_\infty$",
)

# Reference line proportional to h^2.
# It is scaled to pass through the finest-grid point.
reference = max_norms[-1] * (h_values / h_values[-1])**2

ax.loglog(
    h_values,
    reference,
    "--",
    label=r"reference $O(h^2)$",
)

ax.set_xlabel(r"representative grid spacing $h=\max(\Delta x,\Delta y)$")
ax.set_ylabel(r"maximum residual norm $\|R\|_\infty$")
ax.set_title("Convergence of the finite-difference residual")

ax.grid(True, which="both", alpha=0.3)
ax.legend()

# Smaller h means a finer grid, so this makes refinement go visually to the right.
ax.invert_xaxis()

fig.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Point 9
# Estimate observed order from neighbouring error ratios
# ------------------------------------------------------------

orders = np.log(max_norms[:-1] / max_norms[1:]) / np.log(
    h_values[:-1] / h_values[1:]
)

print()
print("Observed orders from max residual norm:")
for (nx1, ny1), (nx2, ny2), order in zip(sizes[:-1], sizes[1:], orders):
    print(
        f"({nx1:4d}, {ny1:4d}) -> ({nx2:4d}, {ny2:4d}): "
        f"observed order = {order:.3f}"
    )

Interpretation:
The residual map shows the local discretization error of the finite-difference equation on the interior grid points.
The log-log plot should follow the reference O(h^2) line, and the observed orders should be close to 2.
This agrees with the expected second-order accuracy of the centred finite-difference stencil for the second derivative.

Optional extension comment:
For higher wavenumbers, for example k = 5 and ell = 7, the formal order of the centred finite-difference stencil should still be close to second order.
What changes mainly is the error constant: more oscillatory functions have larger higher derivatives, so the same grid spacing produces a larger residual.
In practice, the convergence line should remain roughly parallel to O(h^2), but it will be shifted upward.